In [ ]:
import importlib
import hydra
import torch
from omegaconf import OmegaConf
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import importlib
import sys
import os

mpl.rcParams['xtick.labelsize'] = 12
mpl.rcParams['ytick.labelsize'] = 12

In [ ]:



model_name = "4D_baseline_to_limited_or"
relative_path = os.path.join('..', '..', 'dptorch')

notebook_dir = os.getcwd()
absolute_path = os.path.abspath(os.path.join(notebook_dir, relative_path))

sys.path.insert(0, absolute_path)

def _iter_num(p):
    try:
        return int(p.stem.split("_")[-1])
    except ValueError:
        return -1
import pathlib

list_all_Iter=list(pathlib.Path(os.path.abspath(f"data/4D/baseline")).rglob("*.pth"))
all_checkpoint_iters_bl = [_iter_num(p) for p in list_all_Iter]

latest_checkpoint_num_bl = max(all_checkpoint_iters_bl)

stored_checkpoint_file_bl = 3787
checkpoint_file_bl = latest_checkpoint_num_bl


model = importlib.import_module(f"{model_name}.Model")

torch.manual_seed(123)


m_4d = model.SpecifiedModel.load(
    path=os.path.abspath(f"data/4D/baseline/Iter_{checkpoint_file_bl}.pth"),
    cfg_override={"distributed": False, "init_with_zeros": False, "MODEL_NAME": model_name},
)


simulation_4d = np.loadtxt(hydra.utils.to_absolute_path(f"data/4D/baseline/simulation_{checkpoint_file_bl}.txt"))

In [ ]:
n_types  = m_4d.cfg["model"]["params"]["n_types"]

disc_states = (simulation_4d[:,n_types]).astype(np.int64)
promise_util_4d = np.zeros((simulation_4d.shape[0]-1,))
consumption_4d = np.zeros((simulation_4d.shape[0]-1,))
indx_c = m_4d.P["c_1"]
upper_c = m_4d.cfg["model"]["params"]["upper_trans"]

# disc_states_or = (simulation_4d_or[:,n_types]).astype(np.int64)
# promise_util_4d_or = np.zeros((simulation_4d_or.shape[0]-1,))
# consumption_4d_or = np.zeros((simulation_4d_or.shape[0]-1,))
# indx_c_or = m_4d_or.P["c_1"]

run_in_time = 100
for indx in range(simulation_4d.shape[0]-1):
    promise_util_4d[indx] = simulation_4d[indx,disc_states[indx+1]]
    consumption_4d[indx] = simulation_4d[indx,n_types + 5 +  indx_c + disc_states[indx+1]]
    # promise_util_4d_or[indx] = simulation_4d_or[indx,disc_states_or[indx+1]]
    # consumption_4d_or[indx] = np.minimum(upper_c,simulation_4d_or[indx,n_types + 5 +  indx_c_or + disc_states_or[indx+1]])    

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
beta  = m_4d.cfg["model"]["params"]["beta"]
# Adjust spacing between plots
fig.tight_layout(pad=4.0)

# Plotting the histogram
axes[0].hist(promise_util_4d[run_in_time:], bins=10, edgecolor='black',label="baseline", alpha=0.5)
# axes[0].hist(promise_util_4d_or[run_in_time:], bins=15, edgecolor='black',label="over-reporting", alpha=0.5)

# Adding labels and title
axes[0].set_xlabel('Promise utility', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Frequency', fontsize=14, fontweight='bold')
# plt.title('Histogram of NumPy Vector')
axes[0].legend(fontsize=12)

# Plotting the histogram
axes[1].hist(consumption_4d[run_in_time:], bins=10, edgecolor='black',label="baseline", alpha=0.5)
# axes[1].hist(consumption_4d_or[run_in_time:], bins=10, edgecolor='black',label="over-reporting", alpha=0.5)

# Adding labels and title
axes[1].set_xlabel('Consumption', fontsize=14, fontweight='bold')
# axes[1].set_ylabel('Frequency', fontsize=14)
# plt.legend()


In [ ]:
model_name = "4D_limited_overreporting"
relative_path = os.path.join('..', '..', 'dptorch')

notebook_dir = os.getcwd()
absolute_path = os.path.abspath(os.path.join(notebook_dir, relative_path))

sys.path.insert(0, absolute_path)

def _iter_num(p):
    try:
        return int(p.stem.split("_")[-1])
    except ValueError:
        return -1
import pathlib

list_all_Iter=list(pathlib.Path(os.path.abspath(f"data/4D/overreporting")).rglob("*.pth"))
all_checkpoint_iters_or = [_iter_num(p) for p in list_all_Iter]

latest_checkpoint_num_or = max(all_checkpoint_iters_or)

stored_checkpoint_file_or = 2999
checkpoint_file_or = latest_checkpoint_num_or


model = importlib.import_module(f"{model_name}.Model")



# RNG
torch.manual_seed(123)


m_4d_or = model.SpecifiedModel.load(
    path=os.path.abspath(f"data/4D/overreporting/Iter_{checkpoint_file_or}.pth"),
    cfg_override={"distributed": False, "init_with_zeros": False, "MODEL_NAME": model_name},
)

simulation_4d_or = np.loadtxt(hydra.utils.to_absolute_path(f"data/4D/overreporting/simulation_{checkpoint_file_or}.txt"))


In [ ]:
n_types  = m_4d_or.cfg["model"]["params"]["n_types"]

disc_states_4d_or = (simulation_4d_or[:,n_types]).astype(np.int64)
promise_util_4d_or = np.zeros((simulation_4d_or.shape[0]-1,))
consumption_4d_or = np.zeros((simulation_4d_or.shape[0]-1,))
indx_u_4d_or = m_4d_or.P["u_1"]
indx_c_4d_or = m_4d_or.P["c_1"]
upper_c_4d_or = m_4d_or.cfg["model"]["params"]["upper_trans"]
shock_vec=m_4d_or.cfg["model"]["params"]["shock_vec"]
reg_c=m_4d_or.cfg["model"]["params"]["reg_c"]

run_in_time = 100
overreporting_active = 0
for indx in range(simulation_4d_or.shape[0]-1):
    promise_util_4d_or[indx] = simulation_4d_or[indx,disc_states_4d_or[indx+1]]
    # consumption_4d_or[indx] = (simulation_4d_or[indx,n_types + 5 +  indx_u_4d_or + disc_states_4d_or[indx+1]])**2 - reg_c
    consumption_4d_or[indx] = simulation_4d_or[indx,n_types + 5 +  indx_c_4d_or + disc_states_4d_or[indx+1]]
    tmp_overreporting_active = 0
    for indx_true in range(1,n_types): # true state in
        for indx_false in range(indx_true,n_types): # false state in
            # if shock_vec[indx_true] + (simulation_4d_or[indx,n_types + 5 +  indx_u_4d_or + indx_false])**2 - reg_c - shock_vec[indx_false] > 0:
            if shock_vec[indx_true] + simulation_4d_or[indx,n_types + 5 +  indx_c_4d_or + indx_false] - shock_vec[indx_false] > 0:
                tmp_overreporting_active = 1
                break
    overreporting_active+=tmp_overreporting_active

    # promise_util_4d_or[indx] = simulation_4d_or[indx,disc_states_or[indx+1]]
    # consumption_4d_or[indx] = np.minimum(upper_c,simulation_4d_or[indx,n_types + 5 +  indx_c_or + disc_states_or[indx+1]])    

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
beta  = m_4d.cfg["model"]["params"]["beta"]
# Adjust spacing between plots
fig.tight_layout(pad=4.0)

# Plotting the histogram
# axes[0].hist(promise_util_4d[run_in_time:]*(1-beta), bins=10, edgecolor='black',label="baseline", alpha=0.5)
axes[0].hist(promise_util_4d_or[run_in_time:], bins=15, edgecolor='black',label="overreporting", alpha=0.5)

# Adding labels and title
axes[0].set_xlabel('Promise utility', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Frequency', fontsize=14, fontweight='bold')
# plt.title('Histogram of NumPy Vector')
axes[0].legend(fontsize=12)

# Plotting the histogram
# axes[1].hist(consumption_4d[run_in_time:], bins=10, edgecolor='black',label="baseline", alpha=0.5)
axes[1].hist(consumption_4d_or[run_in_time:], bins=40, edgecolor='black',label="overreporting", alpha=0.5)

# Adding labels and title
axes[1].set_xlabel('Consumption', fontsize=14, fontweight='bold')
# axes[1].set_ylabel('Frequency', fontsize=14)
# plt.legend()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
beta  = m_4d.cfg["model"]["params"]["beta"]
# Adjust spacing between plots
fig.tight_layout(pad=4.0)

# Plotting the histogram
axes[0].hist(promise_util_4d[run_in_time:], bins=10, edgecolor='black',label="baseline", alpha=0.5)
axes[0].hist(promise_util_4d_or[run_in_time:], bins=10, edgecolor='black',label="overreporting", alpha=0.5)

# Adding labels and title
axes[0].set_xlabel('Promise utility', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Frequency', fontsize=14, fontweight='bold')
# plt.title('Histogram of NumPy Vector')
axes[0].legend(fontsize=12)

# Plotting the histogram
axes[1].hist(consumption_4d[run_in_time:], bins=15, edgecolor='black',label="baseline", alpha=0.5)
axes[1].hist(consumption_4d_or[run_in_time:], bins=10, edgecolor='black',label="overreporting", alpha=0.5)

# Adding labels and title
axes[1].set_xlabel('Consumption', fontsize=14, fontweight='bold')
# axes[1].set_ylabel('Frequency', fontsize=14)

# Display the plot
plt.savefig('Figure_7_compare_long_run_histogram_4D.pdf', dpi=500, bbox_inches='tight')


### BAL

In [ ]:
def bayesian_opt_criterion(m, eval_pt,discrete_state, target_p, rho, beta):

    # #compute bayesian optimization criteria
    mean_v = m.M[discrete_state][target_p].predict_mean(
                    eval_pt
                )
    var_v = m.M[discrete_state][target_p].predict_var(
                    eval_pt
                )

    #Deisenroth criterion
    out_vec = (rho * (mean_v) + beta / 2.0 * torch.log(var_v + 1e-15))

    return out_vec

In [ ]:
bal_util = 0
for indxt in range(n_types):
    mask_0 = m_4d.state_sample[:,-1] == indxt
    no_init_samples = m_4d.cfg["no_samples"]
    n_pts = m_4d.state_sample[mask_0,:].shape[0]
    beta =1
    rho=1

    indxp = n_pts-no_init_samples - 1

    train_sample = m_4d.state_sample[mask_0,:][:no_init_samples+indxp,:-1]
    train_v = m_4d.V_sample[mask_0][:no_init_samples+indxp]
    m_4d.M[indxt][0].set_train_data(
        train_sample,
        train_v,
        strict=False,
    )

    bal_util += bayesian_opt_criterion(m_4d,m_4d.state_sample[mask_0,:][no_init_samples+indxp:no_init_samples+indxp+1,:-1],indxt,0,rho,beta) / n_types

In [ ]:
bal_util

In [ ]:
bal_util_or = 0
for indxt in range(n_types):
    mask_0 = m_4d_or.state_sample[:,-1] == indxt
    no_init_samples = m_4d_or.cfg["no_samples"]
    n_pts = m_4d_or.state_sample[mask_0,:].shape[0]
    beta =1
    rho=1

    indxp = n_pts-no_init_samples - 1

    train_sample = m_4d_or.state_sample[mask_0,:][:no_init_samples+indxp,:-1]
    train_v = m_4d_or.V_sample[mask_0][:no_init_samples+indxp]
    m_4d_or.M[indxt][0].set_train_data(
        train_sample,
        train_v,
        strict=False,
    )

    bal_util_or += bayesian_opt_criterion(m_4d_or,m_4d_or.state_sample[mask_0,:][no_init_samples+indxp:no_init_samples+indxp+1,:-1],indxt,0,rho,beta) / n_types

In [ ]:
n_pts

In [ ]:
bal_util_or

### Errors

In [ ]:
error_data_bl = np.loadtxt(f"data/4D/baseline/V_func_error_{checkpoint_file_bl}.txt")
L2_bl = error_data_bl[..., 2]
Linf_bl = error_data_bl[..., 3]

error_data_or = np.loadtxt(f"data/4D/overreporting/V_func_error_{checkpoint_file_or}.txt")
L2_or = error_data_or[..., 2]
Linf_or = error_data_or[..., 3]


sim_data_bl = np.loadtxt(f"data/4D/baseline/simulation_{checkpoint_file_bl}.txt")

sim_data_or = np.loadtxt(f"data/4D/overreporting/simulation_{checkpoint_file_or}.txt")

sim_diff_bl = sim_data_bl[:, 8]
sim_diff_or = sim_data_or[:, 8]


sim_L2_bl = np.sqrt(np.mean(sim_diff_bl**2))
sim_Linf_bl = np.max(sim_diff_bl)
sim_L2_or = np.sqrt(np.mean(sim_diff_or**2))
sim_Linf_or = np.max(sim_diff_or)


def format_latex_sci(value):
    mantissa, exponent = f"{value:.1e}".split("e")
    return rf"${float(mantissa):.1f}\cdot 10^{{{int(exponent)}}}$"


latex_table = f"""

\\begin{{tabular}}{{l|l|c|c}}
    \\hline \\hline
    \\text{{Model Version}} & \\text{{Error type}} & \\text{{$L_2$}} & \\text{{$L_\\infty$}} \\\\
    \\hline \\hline
    Benchmark & Criterion 2 (global error) & {{{format_latex_sci(L2_bl[-1])}}} & {{{format_latex_sci(Linf_bl[-1])}}} \\\\
    Benchmark & Criterion 3 (error along a simulated path) & {{{format_latex_sci(sim_L2_bl)}}} & {{{format_latex_sci(sim_Linf_bl)}}} \\\\
    \\hline
    Overreporting & Criterion 2 (global error) & {{{format_latex_sci(L2_or[-1])}}} & {{{format_latex_sci(Linf_or[-1])}}} \\\\
    Overreporting & Criterion 3 (error along a simulated path) & {{{format_latex_sci(sim_L2_or)}}} & {{{format_latex_sci(sim_Linf_or)}}} \\\\
    \\hline

\\end{{tabular}}

""".strip()


print(latex_table)

with open("Table_8_error_table.tex", "w", encoding="utf-8") as table_file:
    table_file.write(latex_table)
# latex_table

In [ ]:
m_l2 = float(np.mean((m_4d.metrics[list(m_4d.metrics.keys())[-1]]["l2"]).numpy()))
m_inf = float(np.mean((m_4d.metrics[list(m_4d.metrics.keys())[-1]]["l_inf"]).numpy()))
mor_l2 = float(np.mean((m_4d_or.metrics[list(m_4d_or.metrics.keys())[-1]]["l2"]).numpy()))
mor_inf = float(np.mean((m_4d_or.metrics[list(m_4d_or.metrics.keys())[-1]]["l_inf"]).numpy()))

table_path = os.path.join(notebook_dir, "4D_pointwise_error_table.tex")
with open(table_path, "w", encoding="utf-8") as f:
    f.write(
        "\\begin{table}[ht]\n"
        "\\centering\n"
        "\\begin{tabular}{lcc}\n"
        "\\hline\n"
        "4D Model & $L_2$ & $L_\\infty$ \\\\\n"
        "\\hline\n"
        f"Baseline & ${m_l2:.4e}$ & ${m_inf:.4e}$ \\\\\n"
        f"Overreporting & ${mor_l2:.4e}$ & ${mor_inf:.4e}$ \\\\\n"
        "\\hline\n"
        "\\end{tabular}\n"
        "\\end{table}\n"
    )

print(f"Saved LaTeX table to: {table_path}")